# E5 — ConvNeXt-Tiny (HuggingFace) training pilot


## 1. Clone repo asli dan install dependency yang dikunci persis

In [ ]:
import subprocess, sys

REPO_DIR = "/content/diabetic-rethinography"

subprocess.run(["git", "clone", "--branch", "novelty",
                 "https://github.com/salmakamila126/diabetic-rethinography", REPO_DIR], check=True)

# transformers HARUS persis 4.57.3 -- foundation_adaptation.load_local_processor menolak versi lain.
get_ipython().system(f"{sys.executable} -m pip install -q 'transformers==4.57.3' huggingface_hub safetensors")

sys.path.insert(0, REPO_DIR)
print("Repo di-clone ke:", REPO_DIR)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Repo di-clone ke: /content/diabetic-rethinography


In [ ]:
import transformers
assert transformers.__version__ == "4.57.3", f"Dapat {transformers.__version__}, butuh persis 4.57.3"
print("transformers:", transformers.__version__, "OK")

from novelty_integrated.e5 import config as e5_config
from novelty_integrated.e5 import data as e5_data
from novelty_integrated.e5 import runner as e5_runner
from novelty_integrated.e5.model import E5Heads
from novelty import preof_phase1 as phase
from novelty import foundation_adaptation as fa
print("Import modul E5 dari repo asli: OK")


transformers: 4.57.3 OK
Import modul E5 dari repo asli: OK


## 2. Siapkan artifact root dan BUAT manifest baru dari dataset mentah

`ARTIFACT_ROOT/datasets/<DATASET>/` harus sudah berisi gambar mentah APTOS2019/DDR (format sama seperti yang dipakai notebook ConvNeXtV2 Anda -- `train.csv`+`train_images/` untuk APTOS, `DR_grading/{train,valid,test}.txt` untuk DDR). Kalau Anda sudah punya `CAD_ViT/datasets/` dengan data itu, arahkan `ARTIFACT_ROOT` ke situ agar tidak perlu menyalin ulang gambar.

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

ARTIFACT_ROOT = "/content/drive/MyDrive/CAD_ViT"   # ganti ke root yang BERISI datasets/APTOS2019 & datasets/DDR Anda
DATASET = "APTOS2019"        # atau "DDR"
ARM = "integrated"           # "control" | "ordinal" | "deferred" | "integrated"
SPLIT_SEED = 42
OUTPUT_DIR = f"{ARTIFACT_ROOT}/e5_runs/e5_{DATASET.lower()}_{ARM}_seed{SPLIT_SEED}_newsplit"

dataset_root = Path(ARTIFACT_ROOT) / "datasets" / DATASET
if not dataset_root.is_dir():
    raise FileNotFoundError(f"Dataset mentah tidak ditemukan: {dataset_root}")
Path(f"{ARTIFACT_ROOT}/cad_vit_runs/manifests").mkdir(parents=True, exist_ok=True)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)
print("Dataset root:", dataset_root)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ARTIFACT_ROOT: /content/drive/MyDrive/CAD_ViT
Dataset root: /content/drive/MyDrive/CAD_ViT/datasets/APTOS2019


In [ ]:
# --- Pemindaian + validasi gambar mentah (identik metodologis dengan notebook ConvNeXtV2 Anda) ---
import hashlib
import pandas as pd
from PIL import Image, ImageOps

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg"}


def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def validate_records(records, root):
    '''Kolom wajib: image_id, label, image_path (relatif terhadap root).'''
    needed = {"image_id", "label", "image_path"}
    if not needed.issubset(records.columns):
        raise ValueError(f"Kolom wajib: {sorted(needed)}")
    result = records.copy()
    result["image_id"] = result["image_id"].astype("string").str.strip()
    if result["image_id"].isna().any() or result["image_id"].eq("").any():
        raise ValueError("image_id kosong")
    if result["image_id"].duplicated().any():
        raise ValueError("Duplikasi image_id")
    labels = pd.to_numeric(result["label"], errors="coerce")
    if (~labels.isin(range(5))).any():
        raise ValueError("Label harus integer 0..4")
    result["label"] = labels.astype(int)
    root = Path(root).resolve()
    hashes, paths = [], []
    for raw in result["image_path"]:
        relative = Path(str(raw))
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Path gambar harus relatif dan aman: {raw}")
        full = (root / relative).resolve()
        if root not in full.parents or not full.is_file() or full.suffix.lower() not in IMAGE_EXTENSIONS:
            raise FileNotFoundError(f"Gambar hilang/tidak didukung: {raw}")
        with Image.open(full) as im:
            im.verify()
        paths.append(full.relative_to(root).as_posix())
        hashes.append(_sha256_file(full))
    result["image_path"] = paths
    result["sha256"] = hashes
    result["group_id"] = result["sha256"].map(lambda h: "hash:" + h)
    return result[["image_id", "label", "image_path", "group_id", "sha256"]]


def aptos_records(root):
    frame = pd.read_csv(root / "train.csv")
    frame = frame.rename(columns={"id_code": "image_id", "diagnosis": "label"})
    frame["image_path"] = frame["image_id"].map(lambda i: f"train_images/{i}.png")
    return frame[["image_id", "label", "image_path"]]


def ddr_records(root):
    direct = root / "DDR-dataset" / "DR_grading"
    grading_root = direct if direct.is_dir() else next(
        p for p in root.rglob("DR_grading") if p.is_dir()
        and all((p / n).is_file() for n in ("train.txt", "valid.txt", "test.txt")))
    images = {p.name: p for p in grading_root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS}
    rows = []
    for annotation_name in ("train.txt", "valid.txt", "test.txt"):
        with (grading_root / annotation_name).open("r", encoding="utf-8-sig") as stream:
            for line in stream:
                if not line.strip():
                    continue
                filename, raw_label = line.split()
                label = int(raw_label)
                if label not in range(5):
                    continue  # kelas 5 = ungradable, dilewati (sama seperti notebook ConvNeXtV2)
                image_path = images[filename]
                rows.append({"image_id": filename, "label": label,
                             "image_path": image_path.resolve().relative_to(root.resolve()).as_posix()})
    return pd.DataFrame(rows)


raw_records = aptos_records(dataset_root) if DATASET == "APTOS2019" else ddr_records(dataset_root)
validated = validate_records(raw_records, dataset_root)
print(f"Gambar valid untuk {DATASET}:", len(validated))


Gambar valid untuk APTOS2019: 3662


In [ ]:
# --- Split 8:1:1 ter-stratifikasi per kelas, grouping SHA-256 anti-leakage (seed=42) ---
from sklearn.model_selection import train_test_split

def group_split_8_1_1(frame, seed):
    groups = frame.groupby("group_id", sort=True).agg(stratum=("label", "max")).reset_index()
    train_g, held_g = train_test_split(groups, test_size=0.2, random_state=seed, stratify=groups["stratum"])
    val_g, test_g = train_test_split(held_g, test_size=0.5, random_state=seed, stratify=held_g["stratum"])
    allocation = {str(g): part for part, subset in
                  (("train", train_g), ("val", val_g), ("test", test_g)) for g in subset["group_id"]}
    frame = frame.copy()
    frame["split"] = frame["group_id"].astype(str).map(allocation)
    return frame

manifest_frame = group_split_8_1_1(validated, SPLIT_SEED)
print(manifest_frame["split"].value_counts())


split
train    2925
test      369
val       368
Name: count, dtype: int64


In [ ]:
# --- Tulis manifest CSV + sidecar JSON dalam skema yang e5/data.py butuhkan, lalu OVERRIDE hash terkunci ---
import json

output_columns = ["image_id", "image_path", "label", "split", "group_id", "sha256"]
manifest_path = Path(ARTIFACT_ROOT) / "cad_vit_runs" / "manifests" / f"{DATASET}_seed{SPLIT_SEED}.csv"
manifest_frame[output_columns].to_csv(manifest_path, index=False)

actual_hash = phase.sha256(manifest_path)
val_count = int((manifest_frame["split"] == "val").sum())

sidecar = {"dataset": DATASET, "seed": SPLIT_SEED, "manifest_file_sha256": actual_hash,
           "root": str(dataset_root.resolve())}
manifest_path.with_suffix(".json").write_text(json.dumps(sidecar, indent=2), encoding="utf-8")

# Override SPESIFIK untuk sesi notebook ini saja -- TIDAK mengubah file repo. Lihat §0 untuk konsekuensinya.
phase.SPECS[DATASET]["manifest_sha256"] = actual_hash
phase.SPECS[DATASET]["val_count"] = val_count

print(f"Manifest baru ditulis: {manifest_path}")
print(f"Hash (di-override di sesi ini): {actual_hash}")
print(f"val_count (di-override di sesi ini): {val_count}")
print("PERINGATAN: ini BUKAN manifest beku asli proyek -- hasil run ini tidak sebanding "
      "dengan baseline B1/B2 yang tercatat di preof_phase1.SPECS.")


Manifest baru ditulis: /content/drive/MyDrive/CAD_ViT/cad_vit_runs/manifests/APTOS2019_seed42.csv
Hash (di-override di sesi ini): 3b58bc69799cac8016141684c0271779052096f0fe1fb595ee6a9b532f66d042
val_count (di-override di sesi ini): 368
PERINGATAN: ini BUKAN manifest beku asli proyek -- hasil run ini tidak sebanding dengan baseline B1/B2 yang tercatat di preof_phase1.SPECS.


## 3. Unduh dan verifikasi snapshot resmi `facebook/convnext-tiny-224`

Revisi dan hash bobot dikunci di `config.OFFICIAL_SNAPSHOT` -- bukan sesuatu yang bisa diganti.

In [ ]:
import json
from huggingface_hub import snapshot_download

snap = e5_config.OFFICIAL_SNAPSHOT
print("Snapshot resmi yang dikunci:", json.dumps(snap, indent=2))

hf_cache_dir = snapshot_download(
    repo_id=snap["hub_id"], revision=snap["revision"],
    allow_patterns=["config.json", "preprocessor_config.json", snap["weight_file"]],
)
model_dir = Path(hf_cache_dir)
if model_dir.name != snap["revision"]:
    import shutil
    renamed = Path(f"/content/hf_official_models/convnext-tiny-224/{snap['revision']}")
    renamed.parent.mkdir(parents=True, exist_ok=True)
    if renamed.exists():
        shutil.rmtree(renamed)
    shutil.copytree(model_dir, renamed)
    model_dir = renamed

weight_path = model_dir / snap["weight_file"]
actual_weight_hash = phase.sha256(weight_path)
if actual_weight_hash != snap["weight_sha256"]:
    raise ValueError(
        f"Hash bobot snapshot TIDAK cocok -- HuggingFace mungkin sudah mengubah file ini.\n"
        f"  Diharapkan: {snap['weight_sha256']}\n  Didapat: {actual_weight_hash}")
print("Snapshot terverifikasi di:", model_dir)
MODEL_DIR = str(model_dir)


Snapshot resmi yang dikunci: {
  "hub_id": "facebook/convnext-tiny-224",
  "revision": "6166b7613034066690a621d8bf25ffdf181a34f0",
  "config_sha256": "441f982e839a9ea96df7c1678422dc4e65cd5632731a6957f6aa03728195423d",
  "processor_config_sha256": "fd575b890da5a949493e1d1e7a70bfcb9e4b99fe444004d2dbfa253add254741",
  "weight_file": "pytorch_model.bin",
  "weight_sha256": "1fe15b2eba9e355de14c4e244e6365bf61ffef84da7d3442c0afbb2d8564f477"
}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/114M [00:00<?, ?B/s]

Snapshot terverifikasi di: /root/.cache/huggingface/hub/models--facebook--convnext-tiny-224/snapshots/6166b7613034066690a621d8bf25ffdf181a34f0


## 4. Bangun argumen CLI -- dengan caching AKTIF

Caching yang diaktifkan di notebook ini:

| | Training | Validasi |
|---|---|---|
| Caching | **Ya -- RAM** (`DecodedRGB`, per-proses, LRU dibatasi `DECODED_CACHE_MIB`) | **Ya -- disk** (`validation_cache`, tensor resmi ConvNeXt yang sudah diproses) |
| Yang disimpan | **Gambar RGB hasil decode** (sebelum augmentasi) | **Tensor hasil `AutoImageProcessor` resmi** (setelah preprocessing penuh) |
| Augmentasi | **TIDAK di-cache** -- `B1Collator` selalu menjalankan `augment.apply(...)` baru di atas RGB yang di-cache, setiap epoch | tidak relevan (validasi tidak diaugmentasi) |

Ini persis desain `DecodedRGB`/`B1Collator` di `input_pipeline.py`: cache menyimpan RGB mentah hasil decode (verified terhadap SHA-256 sumber), lalu augmentasi training tetap dihitung ulang tiap panggilan -- jadi setiap epoch tetap mendapat augmentasi acak yang baru, hanya langkah decode gambar dari disk yang dihindari pada epoch ke-2 dan seterusnya.

Mengaktifkan salah satu cache memicu `accelerated_input=True` di kode asli, yang MEWAJIBKAN langkah `prepare-cache` (§5) dan `verify-inputs` (§6) dijalankan dulu sebelum `smoke`/`train` -- kode menolak jalan tanpa `--input-preflight-report` yang valid.

In [ ]:
import argparse
import random

import numpy as np
import psutil

# --- Ukur rata-rata ukuran RGB decoded (resolusi ASLI, sebelum resize processor) dari sampel ---
train_paths = manifest_frame.loc[manifest_frame["split"] == "train", "image_path"].tolist()
sample_n = min(80, len(train_paths))
sample_paths = random.Random(42).sample(train_paths, sample_n)
decoded_bytes = []
for p in sample_paths:
    with Image.open(dataset_root / p) as im:
        rgb = ImageOps.exif_transpose(im).convert("RGB")
        decoded_bytes.append(np.asarray(rgb).nbytes)
avg_decoded_bytes = sum(decoded_bytes) / len(decoded_bytes)
train_count = len(train_paths)
full_train_mib = avg_decoded_bytes * train_count / (1024 ** 2)

# --- RAM SISTEM yang benar-benar tersedia sekarang di runtime Colab ini (bukan VRAM L4) ---
available_mib = psutil.virtual_memory().available / (1024 ** 2)
ram_budget_mib = available_mib * 0.60   # sisakan 40% untuk Python/OS/Drive IO/buffer DataLoader

DECODED_CACHE_MIB = int(min(full_train_mib * 1.1, ram_budget_mib))
VALIDATION_CACHE_ROOT = f"{ARTIFACT_ROOT}/e5_runs/val_cache_{DATASET.lower()}_seed{SPLIT_SEED}"

val_count_est = int((manifest_frame["split"] == "val").sum())
val_cache_mib_est = val_count_est * (3 * 224 * 224 * 4 + 4096) / (1024 ** 2)   # lihat validation_cache.build()
CACHE_MAX_GIB = max(1.0, round(val_cache_mib_est * 3 / 1024, 2))               # headroom 3x, tetap kecil
CACHE_RESERVE_GIB = 5.0

print(f"Rata-rata RGB decoded/gambar: {avg_decoded_bytes/1024/1024:.1f} MiB (sampel {sample_n} gambar)")
print(f"Perkiraan seluruh train split decoded: {full_train_mib/1024:.2f} GiB ({train_count} gambar)")
print(f"RAM tersedia saat ini: {available_mib/1024:.1f} GiB -> budget cache (60%): {ram_budget_mib/1024:.1f} GiB")
print(f"DECODED_CACHE_MIB diset ke: {DECODED_CACHE_MIB} MiB ({DECODED_CACHE_MIB/1024:.2f} GiB)")
if DECODED_CACHE_MIB + 1 < full_train_mib:
    coverage = DECODED_CACHE_MIB / full_train_mib * 100
    print(f"PERINGATAN: cache hanya menampung ~{coverage:.0f}% train split -- sisanya di-decode ulang "
          f"tiap epoch (LRU evict). Naikkan RAM runtime Colab (High-RAM) bila ingin cache penuh 100%.")
else:
    print("Cache cukup untuk SELURUH train split -- decode ulang dari disk hanya terjadi di epoch pertama; "
          "epoch ke-2 dst. langsung hit RAM (hanya augmentasi yang dihitung ulang).")
print(f"Estimasi val cache (disk, Drive): {val_cache_mib_est/1024:.3f} GiB -> CACHE_MAX_GIB={CACHE_MAX_GIB}")

def make_args(command, **overrides):
    base = dict(
        dataset=DATASET, arm=ARM, artifact_root=ARTIFACT_ROOT,
        manifest_source_root=None, model_dir=MODEL_DIR,
        revision=e5_config.OFFICIAL_SNAPSHOT["revision"],
        weight_sha256=e5_config.OFFICIAL_SNAPSHOT["weight_sha256"],
        trust_legacy_bin=True,   # weight_file resmi adalah pytorch_model.bin (legacy .bin)
        output_dir=OUTPUT_DIR,
        device="cuda:0", precision="bf16", seed=42,
        micro_batch=32, accumulation=1,
        validation_cache_root=VALIDATION_CACHE_ROOT,
    )
    if command == "prepare-cache":
        base.update(cache_max_gib=CACHE_MAX_GIB, cache_reserve_gib=CACHE_RESERVE_GIB)
    else:
        # num_workers=0: cache RAM tetap berguna di proses utama (satu instance DecodedRGB
        # hidup sepanjang training, dipakai lintas epoch) -- tidak perlu multiprocessing.
        base.update(num_workers=0, persistent_workers=False, prefetch_factor=2,
                    pin_memory=False, decoded_cache_mib=DECODED_CACHE_MIB)
    if command == "verify-inputs":
        base.update(sample_per_grade=2)
    if command in ("smoke", "train", "profile"):
        base.update(input_preflight_report=None)
    if command == "train":
        base.update(smoke_report=None, resume=False, trust_run_checkpoint=False)
    base.update(overrides)
    base["command"] = command
    return argparse.Namespace(**base)

Rata-rata RGB decoded/gambar: 10.1 MiB (sampel 80 gambar)
Perkiraan seluruh train split decoded: 28.81 GiB (2925 gambar)
RAM tersedia saat ini: 48.7 GiB -> budget cache (60%): 29.2 GiB
DECODED_CACHE_MIB diset ke: 29928 MiB (29.23 GiB)
Cache cukup untuk SELURUH train split -- decode ulang dari disk hanya terjadi di epoch pertama; epoch ke-2 dst. langsung hit RAM (hanya augmentasi yang dihitung ulang).
Estimasi val cache (disk, Drive): 0.208 GiB -> CACHE_MAX_GIB=1.0


## 5. `prepare-cache` -- bangun cache disk tensor validasi (sekali saja, CPU-only, tanpa model)

Ini membuat `VALIDATION_CACHE_ROOT` dan mengisinya dengan tensor float32 hasil `AutoImageProcessor` resmi untuk setiap gambar validasi, diverifikasi exact-equal terhadap preprocessing tanpa-cache (singleton vs batch vs round-trip). Sel ini aman dijalankan ulang -- bila cache sudah lengkap, langsung dilewati.

In [ ]:
cache_summary_path = Path(VALIDATION_CACHE_ROOT) / "summary.json"
already_complete = cache_summary_path.is_file() and json.loads(
    cache_summary_path.read_text(encoding="utf-8")).get("status") == "COMPLETE"

if already_complete:
    print("Cache validasi sudah lengkap, dilewati:", VALIDATION_CACHE_ROOT)
else:
    prepare_cache_args = make_args("prepare-cache")
    e5_runner.prepare_cache(prepare_cache_args)
    print("Cache validasi dibuat di:", VALIDATION_CACHE_ROOT)

{
  "schema": "e5-validation-official-input-v1",
  "status": "COMPLETE",
  "row_count": 368,
  "metadata_sha256": "91fc51de8b08b43256ca1a0dfd6967dc30a8e4604b22fa302e5a367c0a467ec1",
  "disk_bytes": 221788640,
  "ordered_entry_digest": "39347069b19a3840af5dee25174c284c343e8308d8d75f8e605cd68ce574d911",
  "equivalence": "exact batch/singleton/roundtrip float32 tensor equality"
}
Cache validasi dibuat di: /content/drive/MyDrive/CAD_ViT/e5_runs/val_cache_aptos2019_seed42


## 6. `verify-inputs` -- bukti ekuivalensi CPU-only (RAM cache training + disk cache validasi vs tanpa-cache)

Wajib dijalankan sebelum `smoke`/`train` begitu ada cache yang aktif -- `_prepare()` di `runner.py` menolak berjalan tanpa laporan ini (`input_preflight_report`). Direktori output HARUS baru setiap kali (kode menolak menimpa) -- sel ini otomatis memakai nama baru bila yang lama sudah ada.

In [ ]:
verify_output_dir = f"{OUTPUT_DIR}_verify_inputs"
suffix = 1
while Path(verify_output_dir).exists():
    suffix += 1
    verify_output_dir = f"{OUTPUT_DIR}_verify_inputs_{suffix}"

verify_args = make_args("verify-inputs", output_dir=verify_output_dir)
e5_runner.verify_inputs(verify_args)
INPUT_PREFLIGHT_REPORT = f"{verify_output_dir}/input_equivalence.json"
print("Input preflight report:", INPUT_PREFLIGHT_REPORT)

{
  "schema": "convnext-e5-integrated-v1",
  "status": "ACTUAL_INPUT_EQUIVALENCE_PASS",
  "dataset": "APTOS2019",
  "manifest_sha256": "3b58bc69799cac8016141684c0271779052096f0fe1fb595ee6a9b532f66d042",
  "processor_sha256": "fd575b890da5a949493e1d1e7a70bfcb9e4b99fe444004d2dbfa253add254741",
  "source_sha256": {
    "novelty/foundation_adaptation.py": "2b011e422c40f52a03a3d0aa89e0d18285875da1946058c265cc07747a61f300",
    "novelty/foundation_augmentation.py": "cd7cedcbadda52aff69ef9096f24803703d1ec7b1b90dde19d5bdbc41c8c8ea3",
    "novelty/foundation_input_cache.py": "e135034bb06796e88996b4e80e0b1403219b6616be077ad0b0bab54c8041f4c1",
    "novelty/foundation_reporting.py": "5a62c4fea17566cd9d2111c6191c629ef7f02fdcafe5232a6a013b14b792ed1d",
    "novelty/preof_phase1.py": "fb91f6cdae2e0efe415cd713d607c6b6421b950500b4c1da60ce0fafc66b587d",
    "novelty/preof_e1.py": "e704d1b235683f41d6e8221dc72bfb2cf9050c75bff459c4f506901fdf5576dd",
    "novelty_integrated/e5/config.py": "165b0ad6a0121d7d23

## 7. `smoke` -- dua langkah optimizer, TANPA checkpoint (sanity check wajib sebelum `train`)

In [ ]:
smoke_output_dir = f"{OUTPUT_DIR}_smoke"
smoke_args = make_args("smoke", output_dir=smoke_output_dir, input_preflight_report=INPUT_PREFLIGHT_REPORT)
e5_runner.smoke(smoke_args)
SMOKE_REPORT_PATH = f"{smoke_output_dir}/smoke.json"
print("Smoke report:", SMOKE_REPORT_PATH)

{
  "status": "E5_SMOKE_PASS",
  "dataset": "APTOS2019",
  "arm": "integrated",
  "gpu_peak_reserved_bytes": 2627731456
}
Smoke report: /content/drive/MyDrive/CAD_ViT/e5_runs/e5_aptos2019_integrated_seed42_newsplit_smoke/smoke.json


## 8. `train` -- pilot 20 epoch terkunci (`PILOT_EPOCHS`), menghasilkan `best.pt`/`last.pt`

Membutuhkan `smoke_report` dari §7 yang PERSIS cocok (`_check_smoke` memverifikasi `record` di dalamnya identik dengan konfigurasi run ini -- bila Anda mengubah `DATASET`/`ARM`/`DECODED_CACHE_MIB`/dll setelah menjalankan smoke, jalankan ulang §6-§7 dulu).

In [ ]:
train_args = make_args("train", smoke_report=SMOKE_REPORT_PATH, input_preflight_report=INPUT_PREFLIGHT_REPORT)
e5_runner.train(train_args)
print("\nRun tersimpan di:", OUTPUT_DIR)
print("  best.pt, last.pt, history.json, history.csv, run.json")
print("  summary.json (hanya dibuat bila ke-20 epoch pilot selesai)")

2026-11-20T07:22:37+00:00 epoch=1 phase=train batch=5/92 images=160/2925 loss=1.618788 throughput=2.53 img/s eta=1094s
2026-11-20T07:23:50+00:00 epoch=1 phase=train batch=10/92 images=320/2925 loss=1.416805 throughput=2.35 img/s eta=1108s
2026-11-20T07:25:03+00:00 epoch=1 phase=train batch=15/92 images=480/2925 loss=1.365644 throughput=2.29 img/s eta=1068s
2026-11-20T07:26:05+00:00 epoch=1 phase=train batch=19/92 images=608/2925 loss=1.295533 throughput=2.24 img/s eta=1034s
2026-11-20T07:27:07+00:00 epoch=1 phase=train batch=24/92 images=768/2925 loss=1.231945 throughput=2.30 img/s eta=936s
2026-11-20T07:28:07+00:00 epoch=1 phase=train batch=28/92 images=896/2925 loss=1.196827 throughput=2.28 img/s eta=892s
2026-11-20T07:29:10+00:00 epoch=1 phase=train batch=32/92 images=1024/2925 loss=1.167861 throughput=2.25 img/s eta=847s
2026-11-20T07:30:11+00:00 epoch=1 phase=train batch=36/92 images=1152/2925 loss=1.139434 throughput=2.22 img/s eta=797s
2026-11-20T07:31:22+00:00 epoch=1 phase=tra

In [ ]:
# --- Opsional: lihat statistik hit/miss cache setelah training (butuh akses ke objek `loaders`
#     dari dalam train() -- tidak diexpose runner.py secara default). Statistik cache per-epoch
#     sudah otomatis tercatat di history.json/history.csv kolom-kolom resource bila loaders aktif.
import json as _json
history = _json.loads(Path(f"{OUTPUT_DIR}/history.json").read_text(encoding="utf-8"))
for row in history:
    counts = row["resource"].get("input_cache_counts")
    if counts:
        print(f"epoch {row['epoch']}: {counts}")


## 9. Structured Pruning -- L1-norm, ratio sweep (RQ2, metode sama dengan notebook ConvNeXtV2 §10)

Metode persis sama dengan §10 di `ConvNeXtV2_Tiny_Baseline.ipynb` -- L1-norm pada dimensi ekspansi 4x tiap block, sweep 10/30/50%, fine-tune recovery singkat, lalu CPU benchmark multi-backend. Yang BERBEDA (arsitektur ConvNeXt V1/HuggingFace, bukan ConvNeXtV2/timm):

| | ConvNeXtV2 (`timm`, notebook lain) | ConvNeXt V1 (`transformers`, notebook ini) |
|---|---|---|
| Jalur block | `block.mlp.fc1 -> mlp.grn -> mlp.fc2` | `layer.pwconv1 -> act -> layer.pwconv2` (TANPA GRN) |
| Lokasi block | `model.stages[i].blocks[j]` | `backbone.encoder.stages[i].layers[j]` |
| Checkpoint | satu `state["model"]` gabungan | `state["backbone"]` + `state["heads"]` TERPISAH |
| Loss untuk fine-tune | `nn.CrossEntropyLoss` manual | fungsi ASLI repo `e5_runner._train_batches`/`_eval_batches` (menangani arm ordinal/class-weight otomatis) |
| Kualitas dibandingkan ke | `test_metrics.json` (Stage 1 test split) | **validation** split -- E5 TIDAK PERNAH mengevaluasi test split (`final_test_status` selalu `NOT_ACCESSED`); baseline pembanding diambil dari `best["validation"]` yang sudah tersimpan di `best.pt` itu sendiri. |

Fungsi pruning di bawah **sudah diverifikasi langsung** terhadap `transformers.ConvNextModel` asli (bukan mock) sebelum dikirim -- forward pass tetap berjalan dan `pooler_output`/`last_hidden_state` tetap berbentuk sama setelah pruning, karena yang dipangkas hanya dimensi tersembunyi 4x internal tiap block (tidak menyentuh dimensi `dim` input/output block).

In [ ]:
import copy

import torch
from torch import nn

def _l1_importance_v1(weight):
    '''L1-norm tiap baris (= satu channel ekspansi) dari weight matrix pwconv1.'''
    return weight.detach().abs().sum(dim=1)


def prune_convnext_v1_expansion(backbone, keep_ratio, importance="l1"):
    '''Pangkas dimensi ekspansi 4x (pwconv1 -> act -> pwconv2) di SETIAP ConvNextLayer HF, in-place,
    di seluruh backbone.encoder.stages. keep_ratio = fraksi channel ekspansi yang DIPERTAHANKAN.
    TIDAK ada GRN untuk dipangkas (spesifik ConvNeXtV2) -- layer_scale_parameter (shape [dim]) juga
    TIDAK terpengaruh karena beroperasi pada dimensi output block (dim), bukan dimensi ekspansi (4*dim).'''
    if importance != "l1":
        raise NotImplementedError(f"importance={importance!r} belum diimplementasikan")
    summary = []
    for stage_idx, stage in enumerate(backbone.encoder.stages):
        for layer_idx, layer in enumerate(stage.layers):
            hidden_before = layer.pwconv1.out_features
            keep_n = max(1, round(hidden_before * keep_ratio))
            scores = _l1_importance_v1(layer.pwconv1.weight)
            keep_idx = torch.argsort(scores, descending=True)[:keep_n].sort().values

            new_pwconv1 = nn.Linear(layer.pwconv1.in_features, keep_n,
                                    bias=layer.pwconv1.bias is not None)
            new_pwconv1.weight.data = layer.pwconv1.weight.data[keep_idx].clone()
            if layer.pwconv1.bias is not None:
                new_pwconv1.bias.data = layer.pwconv1.bias.data[keep_idx].clone()

            new_pwconv2 = nn.Linear(keep_n, layer.pwconv2.out_features,
                                    bias=layer.pwconv2.bias is not None)
            new_pwconv2.weight.data = layer.pwconv2.weight.data[:, keep_idx].clone()
            if layer.pwconv2.bias is not None:
                new_pwconv2.bias.data = layer.pwconv2.bias.data.clone()  # dimensi output (dim) tidak berubah

            layer.pwconv1 = new_pwconv1
            layer.pwconv2 = new_pwconv2
            summary.append({"stage": stage_idx, "layer": layer_idx,
                            "hidden_before": hidden_before, "hidden_after": keep_n})
    return pd.DataFrame(summary)


## 10. Harness benchmark CPU (diadaptasi untuk backbone HF ConvNeXt V1 + `E5Heads`)

Notebook ConvNeXtV2 Anda punya harness §9 sendiri, tapi itu di KERNEL TERPISAH -- fungsi di bawah adalah versi yang disesuaikan untuk arsitektur ini (dua modul `backbone`+`heads`, bukan satu `nn.Module` timm), dengan filosofi SAMA: sampel timing (kecil, diulang `n_repeats`x) dipisah dari sampel kualitas (lebih besar, satu pass), diambil dari **validation split** (bukan test).

In [ ]:
import time
from torch.nn import functional as F

class E5InferenceModel(nn.Module):
    '''Bungkus backbone+heads jadi satu modul agar bisa di-trace/export (TorchScript/ONNX)
    sebagai satu graph. Hanya logit kategorikal (grade DR) -- sama seperti E5Heads.predict_logits.'''
    def __init__(self, backbone, heads):
        super().__init__()
        self.backbone = backbone
        self.heads = heads

    def forward(self, pixel_values):
        pooled = self.backbone(pixel_values=pixel_values).pooler_output
        return self.heads.predict_logits(pooled)


def _build_fixed_val_batches(rows_val, processor, n_samples, batch_size=1, seed=42):
    selected = rows_val if n_samples >= len(rows_val) else \
        [rows_val[i] for i in np.random.default_rng(seed).choice(len(rows_val), n_samples, replace=False)]
    x_batches, y_batches = [], []
    for start in range(0, len(selected), batch_size):
        chunk = selected[start:start + batch_size]
        pixels, labels = fa._image_batch(chunk, processor)
        x_batches.append(pixels)
        y_batches.append(labels)
    return x_batches, y_batches, len(selected)


def benchmark_pytorch_cpu(model, x_batches, n_warmup, n_repeats):
    model = model.to("cpu").eval()
    with torch.inference_mode():
        for _ in range(n_warmup):
            for x in x_batches:
                model(x)
        latencies = []
        for _ in range(n_repeats):
            started = time.perf_counter()
            for x in x_batches:
                model(x)
            latencies.append((time.perf_counter() - started) / len(x_batches) * 1000)
    latencies = np.asarray(latencies)
    return {"median_latency_ms_per_image": float(np.median(latencies)),
            "p90_latency_ms_per_image": float(np.percentile(latencies, 90)),
            "throughput_images_per_second": float(1000.0 / np.median(latencies))}


def classification_quality_pytorch(model, x_batches, y_batches):
    model = model.to("cpu").eval()
    truths, probabilities = [], []
    with torch.inference_mode():
        for x, y in zip(x_batches, y_batches):
            logits = model(x)
            probabilities.append(logits.float().softmax(1).numpy())
            truths.append(y.numpy())
    return fa._classification(np.concatenate(truths), np.concatenate(probabilities))


def export_torchscript(model, path):
    model = model.to("cpu").eval()
    example = torch.randn(1, 3, 224, 224)
    traced = torch.jit.trace(model, example)
    traced.save(str(path))


def benchmark_torchscript_cpu(ts_path, x_batches, n_warmup, n_repeats):
    ts_model = torch.jit.load(str(ts_path))
    ts_model.eval()
    return benchmark_pytorch_cpu(ts_model, x_batches, n_warmup, n_repeats)


def classification_quality_torchscript(ts_path, x_batches, y_batches):
    ts_model = torch.jit.load(str(ts_path))
    ts_model.eval()
    return classification_quality_pytorch(ts_model, x_batches, y_batches)


def export_onnx(model, path):
    model = model.to("cpu").eval()
    example = torch.randn(1, 3, 224, 224)
    torch.onnx.export(model, example, str(path), input_names=["pixel_values"],
                      output_names=["logits"], opset_version=17, dynamo=False)


def benchmark_onnxruntime_cpu(onnx_path, x_batches, n_warmup, n_repeats, cpu_threads):
    import onnxruntime as ort
    options = ort.SessionOptions()
    options.intra_op_num_threads = cpu_threads
    session = ort.InferenceSession(str(onnx_path), options, providers=["CPUExecutionProvider"])
    for _ in range(n_warmup):
        for x in x_batches:
            session.run(None, {"pixel_values": x.numpy()})
    latencies = []
    for _ in range(n_repeats):
        started = time.perf_counter()
        for x in x_batches:
            session.run(None, {"pixel_values": x.numpy()})
        latencies.append((time.perf_counter() - started) / len(x_batches) * 1000)
    latencies = np.asarray(latencies)
    return {"median_latency_ms_per_image": float(np.median(latencies)),
            "p90_latency_ms_per_image": float(np.percentile(latencies, 90)),
            "throughput_images_per_second": float(1000.0 / np.median(latencies))}, session


def classification_quality_onnx(session, x_batches, y_batches):
    truths, probabilities = [], []
    for x, y in zip(x_batches, y_batches):
        logits = session.run(None, {"pixel_values": x.numpy()})[0]
        probabilities.append(torch.from_numpy(logits).float().softmax(1).numpy())
        truths.append(y.numpy())
    return fa._classification(np.concatenate(truths), np.concatenate(probabilities))


def file_size_mb(path):
    return Path(path).stat().st_size / (1024 ** 2)


def compare_against_baseline(baseline_validation_metrics, quality, tolerance):
    rows = []
    for key in ("macro_f1", "weighted_f1", "quadratic_weighted_kappa", "accuracy"):
        base_v = baseline_validation_metrics.get(key)
        new_v = quality.get(key)
        if base_v is None or new_v is None:
            continue
        rows.append({"metric": key, "baseline_validation": base_v, "pruned": new_v,
                    "delta": new_v - base_v, "within_tolerance": abs(new_v - base_v) <= tolerance})
    return pd.DataFrame(rows)

## 11. Konfigurasi sweep pruning dan orkestrasi (prune -> fine-tune recovery -> benchmark)

Fine-tune recovery memakai fungsi training/eval ASLI repo (`e5_runner._train_batches`/`_eval_batches`) -- bukan loop buatan sendiri -- agar perilaku loss (ordinal/class-weight per arm) identik dengan pilot aslinya. **Implementation assumption**: recovery TIDAK memakai `weights_for_epoch` (class weighting arm `deferred`/`integrated` yang baru aktif di epoch >=11 pada pilot 20-epoch) karena recovery jauh lebih singkat (`finetune_epochs` kecil) -- ini murni pemulihan ringan, bukan reproduksi pilot penuh.

In [ ]:
import wandb

wandb.login()

WANDB_PROJECT = "convnextv2-dr-baseline"
print("wandb login OK, project:", WANDB_PROJECT)

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: oraclep8 (oraclep8-brawijaya-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb login OK, project: convnextv2-dr-baseline


In [ ]:
from dataclasses import dataclass, field


@dataclass
class E5PruningConfig:
    prune_ratios: tuple = (0.10, 0.30, 0.50)        # fraksi channel ekspansi yang DIBUANG
    finetune_epochs: int = 3                         # recovery singkat, BUKAN training dari awal
    finetune_backbone_lr: float = e5_config.BACKBONE_LR
    finetune_head_lr: float = e5_config.HEAD_LR
    finetune_weight_decay: float = e5_config.WEIGHT_DECAY
    max_grad_norm: float = e5_config.MAX_GRAD_NORM
    seed: int = 42
    cpu_threads: int = 4
    max_eval_samples: int = 30          # sampel TIMING CPU benchmark (val split)
    quality_eval_samples: int = 150     # sampel KUALITAS CPU benchmark, terpisah dari timing
    n_warmup: int = 10
    n_repeats: int = 20
    metric_tolerance: float = 0.02


def run_e5_pruning_sweep(config: E5PruningConfig):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    precision = fa._precision(device, "bf16" if device.type == "cuda" else "fp32")
    print(f"Fine-tuning recovery akan berjalan di: {device} ({precision})")

    state = fa._load_checkpoint(f"{OUTPUT_DIR}/best.pt", trust=True)
    base_record = state["record"]
    base_validation = state["best"]["validation"]
    train_counts = base_record["train_counts"]
    arm_ordinal, _ = e5_config.arm_components(ARM)
    print(f"Baseline (best.pt, epoch={state['epoch']}): "
          f"val_macro_f1={base_validation['macro_f1']:.4f} val_qwk={base_validation['quadratic_weighted_kappa']:.4f}")

    rows, _ = e5_data.load_train_val(ARTIFACT_ROOT, DATASET,
                                     manifest_source_root=None, verify_images=True)

    snapshot = fa.audit_snapshot("convnext_tiny", MODEL_DIR,
                                 e5_config.OFFICIAL_SNAPSHOT["revision"],
                                 e5_config.OFFICIAL_SNAPSHOT["weight_sha256"])
    pruning_root = Path(f"{OUTPUT_DIR}_pruned")
    pruning_root.mkdir(parents=True, exist_ok=True)

    all_results = {}
    for prune_ratio in config.prune_ratios:
        torch.manual_seed(config.seed)
        print(f"\n########## Pruning ratio: {prune_ratio:.0%} (keep {1 - prune_ratio:.0%} channel ekspansi) ##########")

        backbone, processor = fa.load_local_backbone(snapshot, trust_legacy_bin=True)
        backbone.load_state_dict(state["backbone"])
        heads = E5Heads(768, train_counts, ordinal=arm_ordinal)
        heads.load_state_dict(state["heads"])
        backbone.to(device)
        heads.to(device)

        params_before = sum(p.numel() for p in backbone.parameters())
        prune_summary = prune_convnext_v1_expansion(backbone, keep_ratio=1 - prune_ratio)
        params_after = sum(p.numel() for p in backbone.parameters())
        print(f"Parameter backbone: {params_before:,} -> {params_after:,} "
              f"({(1 - params_after / params_before):.2%} reduction)")

        pruned_name = f"e5_{DATASET.lower()}_{ARM}_pruned_l1_{int(round(prune_ratio * 100))}pct"
        pruned_dir = pruning_root / pruned_name
        pruned_dir.mkdir(parents=True, exist_ok=True)

        # --- Evaluasi SEBELUM fine-tuning (dampak langsung pruning), full validation split ---
        pre_val_batches = fa._validation_batches("full", rows, processor, None, 32)
        pre_ft = e5_runner._eval_batches(backbone, heads, pre_val_batches, device=device,
                                         precision=precision, total_images=len(rows["val"]))
        print(f"Sebelum fine-tuning -- val Macro-F1: {pre_ft['macro_f1']:.4f}  QWK: {pre_ft['quadratic_weighted_kappa']:.4f}")
        wandb.log({"prune_ratio": prune_ratio, "finetune_epoch": 0,
                  "params_before": params_before, "params_after": params_after,
                  "val/macro_f1": pre_ft["macro_f1"], "val/qwk": pre_ft["quadratic_weighted_kappa"],
                  "val/accuracy": pre_ft["accuracy"], "stage": "pre_finetune"})

        # --- Fine-tuning recovery singkat memakai fungsi ASLI repo ---
        optimizer = torch.optim.AdamW([
            {"params": list(backbone.parameters()), "lr": config.finetune_backbone_lr},
            {"params": list(heads.parameters()), "lr": config.finetune_head_lr},
        ], weight_decay=config.finetune_weight_decay)
        best_val_f1, best_backbone_state, best_heads_state = pre_ft["macro_f1"], \
            copy.deepcopy(backbone.state_dict()), copy.deepcopy(heads.state_dict())
        for epoch in range(1, config.finetune_epochs + 1):
            train_batches = fa._epochs_batches("full", rows, processor, None, 32,
                                               config.seed, epoch, experiment="B1")
            train_info = e5_runner._train_batches(
                backbone, heads, optimizer, train_batches, device=device, precision=precision,
                total_images=len(rows["train"]), micro_batch=32, accumulation=1, class_weights=None)
            val_batches = fa._validation_batches("full", rows, processor, None, 32)
            val_info = e5_runner._eval_batches(backbone, heads, val_batches, device=device,
                                               precision=precision, total_images=len(rows["val"]))
            print(f"  [finetune] epoch {epoch}: train_loss={train_info['loss']:.4f} "
                  f"val_macro_f1={val_info['macro_f1']:.4f} val_qwk={val_info['quadratic_weighted_kappa']:.4f}")
            wandb.log({"prune_ratio": prune_ratio, "finetune_epoch": epoch,
                      "train/loss": train_info["loss"], "train/macro_f1": train_info["macro_f1"],
                      "val/macro_f1": val_info["macro_f1"], "val/qwk": val_info["quadratic_weighted_kappa"],
                      "val/accuracy": val_info["accuracy"], "stage": "finetune"})
            if val_info["macro_f1"] > best_val_f1:
                best_val_f1 = val_info["macro_f1"]
                best_backbone_state = copy.deepcopy(backbone.state_dict())
                best_heads_state = copy.deepcopy(heads.state_dict())
        backbone.load_state_dict(best_backbone_state)
        heads.load_state_dict(best_heads_state)

        post_val_batches = fa._validation_batches("full", rows, processor, None, 32)
        post_ft = e5_runner._eval_batches(backbone, heads, post_val_batches, device=device,
                                          precision=precision, total_images=len(rows["val"]))
        print(f"Setelah fine-tuning -- val Macro-F1: {post_ft['macro_f1']:.4f}  QWK: {post_ft['quadratic_weighted_kappa']:.4f}")
        wandb.log({"prune_ratio": prune_ratio, "finetune_epoch": config.finetune_epochs,
                  "val/macro_f1": post_ft["macro_f1"], "val/qwk": post_ft["quadratic_weighted_kappa"],
                  "val/accuracy": post_ft["accuracy"], "stage": "post_finetune"})

        # --- Simpan checkpoint pruned (backbone+heads terpisah, sama format dengan best.pt) ---
        fa._save_checkpoint(pruned_dir / "pruned_backbone_heads.pt",
                            {"backbone": backbone.state_dict(), "heads": heads.state_dict(),
                             "prune_ratio": prune_ratio, "arm": ARM, "dataset": DATASET})
        prune_summary.to_csv(pruned_dir / "prune_summary.csv", index=False)
        (pruned_dir / "pruning_config.json").write_text(json.dumps({
            "prune_ratio": prune_ratio, "finetune_epochs": config.finetune_epochs,
            "params_before": params_before, "params_after": params_after,
            "param_reduction_pct": 1 - params_after / params_before,
            "pre_finetune_val_macro_f1": pre_ft["macro_f1"],
            "post_finetune_val_macro_f1": post_ft["macro_f1"],
            "baseline_val_macro_f1": base_validation["macro_f1"],
        }, indent=2), encoding="utf-8")

        # --- Benchmark CPU multi-backend (val split, timing/kualitas sampel terpisah) ---
        bench_model = E5InferenceModel(copy.deepcopy(backbone).to("cpu").eval(),
                                       copy.deepcopy(heads).to("cpu").eval())
        torch.set_num_threads(config.cpu_threads)
        x_batches, y_batches, n_timing = _build_fixed_val_batches(
            rows["val"], processor, config.max_eval_samples, batch_size=1, seed=config.seed)
        qx_batches, qy_batches, n_quality = _build_fixed_val_batches(
            rows["val"], processor, config.quality_eval_samples, batch_size=1, seed=config.seed)

        bench_results = {"backends": {}, "quality_vs_baseline": {}}
        timing = benchmark_pytorch_cpu(bench_model, x_batches, config.n_warmup, config.n_repeats)
        quality = classification_quality_pytorch(bench_model, qx_batches, qy_batches)
        state_path = pruned_dir / "pruned_state_cpu.pt"
        torch.save(bench_model.state_dict(), state_path)
        timing["model_size_mb"] = file_size_mb(state_path)
        bench_results["backends"]["pytorch_cpu_eager"] = {**timing,
            "quality": {k: v for k, v in quality.items() if k not in ("confusion_matrix", "per_grade")}}
        bench_results["quality_vs_baseline"]["pytorch_cpu_eager"] = compare_against_baseline(
            base_validation, quality, config.metric_tolerance).to_dict(orient="records")

        ts_path = pruned_dir / "pruned_model.torchscript.pt"
        export_torchscript(bench_model, ts_path)
        timing = benchmark_torchscript_cpu(ts_path, x_batches, config.n_warmup, config.n_repeats)
        quality = classification_quality_torchscript(ts_path, qx_batches, qy_batches)
        timing["model_size_mb"] = file_size_mb(ts_path)
        bench_results["backends"]["torchscript_cpu"] = {**timing,
            "quality": {k: v for k, v in quality.items() if k not in ("confusion_matrix", "per_grade")}}
        bench_results["quality_vs_baseline"]["torchscript_cpu"] = compare_against_baseline(
            base_validation, quality, config.metric_tolerance).to_dict(orient="records")

        try:
            onnx_path = pruned_dir / "pruned_model.onnx"
            export_onnx(bench_model, onnx_path)
            timing, session = benchmark_onnxruntime_cpu(onnx_path, x_batches, config.n_warmup,
                                                        config.n_repeats, config.cpu_threads)
            quality = classification_quality_onnx(session, qx_batches, qy_batches)
            timing["model_size_mb"] = file_size_mb(onnx_path)
            bench_results["backends"]["onnxruntime_cpu"] = {**timing,
                "quality": {k: v for k, v in quality.items() if k not in ("confusion_matrix", "per_grade")}}
            bench_results["quality_vs_baseline"]["onnxruntime_cpu"] = compare_against_baseline(
                base_validation, quality, config.metric_tolerance).to_dict(orient="records")
        except Exception as exc:
            print(f"  ONNX Runtime backend dilewati (opsional): {exc}")

        (pruned_dir / "cpu_benchmark.json").write_text(json.dumps(bench_results, indent=2), encoding="utf-8")
        summary_rows = [{"backend": name, "median_latency_ms_per_image": r["median_latency_ms_per_image"],
                         "throughput_images_per_second": r["throughput_images_per_second"],
                         "model_size_mb": r["model_size_mb"], "macro_f1": r["quality"]["macro_f1"],
                         "qwk": r["quality"]["quadratic_weighted_kappa"]}
                        for name, r in bench_results["backends"].items()]
        print(f"  timing: {n_timing} sampel x {config.n_repeats} repeats; kualitas: {n_quality} sampel (satu pass)")
        print(pd.DataFrame(summary_rows).sort_values("median_latency_ms_per_image").to_string(index=False))

        for row in summary_rows:
            wandb.log({"prune_ratio": prune_ratio, "stage": "benchmark",
                      f"benchmark/{row['backend']}/median_latency_ms": row["median_latency_ms_per_image"],
                      f"benchmark/{row['backend']}/throughput_ips": row["throughput_images_per_second"],
                      f"benchmark/{row['backend']}/model_size_mb": row["model_size_mb"],
                      f"benchmark/{row['backend']}/macro_f1": row["macro_f1"],
                      f"benchmark/{row['backend']}/qwk": row["qwk"]})
        wandb.log({"prune_ratio": prune_ratio, "stage": "benchmark_table",
                  f"benchmark_table/ratio_{int(round(prune_ratio * 100))}pct":
                      wandb.Table(dataframe=pd.DataFrame(summary_rows))})

        all_results[prune_ratio] = {
            "pruned_name": pruned_name, "params_before": params_before, "params_after": params_after,
            "pre_finetune_val_macro_f1": pre_ft["macro_f1"], "post_finetune_val_macro_f1": post_ft["macro_f1"],
            "post_finetune_val_qwk": post_ft["quadratic_weighted_kappa"], "benchmark": bench_results,
        }
    return all_results

## 12. Menjalankan sweep pruning

In [ ]:
import copy

wandb_prune_run = wandb.init(
    project=WANDB_PROJECT, job_type="pruning_sweep",
    name=f"prune1_{DATASET.lower()}_{ARM}_seed{SPLIT_SEED}",
    group=f"{DATASET}_{ARM}",
    tags=["e5", "convnext-tiny-v1", "pruning", DATASET, ARM],
    config={"dataset": DATASET, "arm": ARM, "seed": SPLIT_SEED})

e5_pruning_config = E5PruningConfig()
try:
    e5_pruning_results = run_e5_pruning_sweep(e5_pruning_config)
finally:
    wandb.finish()

Fine-tuning recovery akan berjalan di: cuda (bf16)
Baseline (best.pt, epoch=17): val_macro_f1=0.7080 val_qwk=0.9050

########## Pruning ratio: 10% (keep 90% channel ekspansi) ##########
Parameter backbone: 27,820,128 -> 25,227,780 (9.32% reduction)
Sebelum fine-tuning -- val Macro-F1: 0.3500  QWK: 0.5027
  [finetune] epoch 1: train_loss=0.5353 val_macro_f1=0.6682 val_qwk=0.8765
  [finetune] epoch 2: train_loss=0.3864 val_macro_f1=0.6416 val_qwk=0.8717
  [finetune] epoch 3: train_loss=0.3403 val_macro_f1=0.6622 val_qwk=0.8928
Setelah fine-tuning -- val Macro-F1: 0.6682  QWK: 0.8765


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/convnext/modeling_convnext.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1731/3694570102.py:80: DeprecationWarning:

  timing: 30 sampel x 20 repeats; kualitas: 150 sampel (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  macro_f1      qwk
  onnxruntime_cpu                    31.527171                     31.718672      96.332890  0.682156 0.890215
  torchscript_cpu                    39.657402                     25.215974      96.585386  0.682156 0.890215
pytorch_cpu_eager                    43.579053                     22.946804      96.322398  0.682156 0.890215

########## Pruning ratio: 30% (keep 70% channel ekspansi) ##########
Parameter backbone: 27,820,128 -> 20,045,970 (27.94% reduction)
Sebelum fine-tuning -- val Macro-F1: 0.1567  QWK: 0.0722
  [finetune] epoch 1: train_loss=0.8944 val_macro_f1=0.4063 val_qwk=0.7411
  [finetune] epoch 2: train_loss=0.6856 val_macro_f1=0.4145 val_qwk=0.7696
  [finetune] epoch 3: train_loss=0.6159 val_macro_f1=0.4501 val_qwk=0.7854
Setelah fine-tuning -- val Macro-F1: 0.4501  QWK: 0.7854


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/convnext/modeling_convnext.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1731/3694570102.py:80: DeprecationWarning:

  timing: 30 sampel x 20 repeats; kualitas: 150 sampel (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  macro_f1      qwk
  onnxruntime_cpu                    26.975041                     37.071306      76.565819  0.427284 0.744395
  torchscript_cpu                    33.838229                     29.552374      76.818419  0.427284 0.744395
pytorch_cpu_eager                    37.857058                     26.415154      76.555430  0.427284 0.744395

########## Pruning ratio: 50% (keep 50% channel ekspansi) ##########
Parameter backbone: 27,820,128 -> 14,867,616 (46.56% reduction)
Sebelum fine-tuning -- val Macro-F1: 0.1314  QWK: 0.0000
  [finetune] epoch 1: train_loss=1.1530 val_macro_f1=0.3097 val_qwk=0.6667
  [finetune] epoch 2: train_loss=0.8813 val_macro_f1=0.3145 val_qwk=0.7095
  [finetune] epoch 3: train_loss=0.8190 val_macro_f1=0.3109 val_qwk=0.7078
Setelah fine-tuning -- val Macro-F1: 0.3145  QWK: 0.7095


/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/jit/_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/models/convnext/modeling_convnext.py:116: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if num_channels != self.num_channels:
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(
/tmp/ipykernel_1731/3694570102.py:80: DeprecationWarning:

  timing: 30 sampel x 20 repeats; kualitas: 150 sampel (satu pass)
          backend  median_latency_ms_per_image  throughput_images_per_second  model_size_mb  macro_f1      qwk
  onnxruntime_cpu                    21.274974                     47.003583      56.811966   0.30026 0.656085
  torchscript_cpu                    26.739548                     37.397790      57.066002   0.30026 0.656085
pytorch_cpu_eager                    30.490950                     32.796617      56.801951   0.30026 0.656085


benchmark/onnxruntime_cpu/macro_f1,█▃▁
benchmark/onnxruntime_cpu/median_latency_ms,█▅▁
benchmark/onnxruntime_cpu/model_size_mb,█▄▁
benchmark/onnxruntime_cpu/qwk,█▄▁
benchmark/onnxruntime_cpu/throughput_ips,▁▃█
benchmark/pytorch_cpu_eager/macro_f1,█▃▁
benchmark/pytorch_cpu_eager/median_latency_ms,█▅▁
benchmark/pytorch_cpu_eager/model_size_mb,█▄▁
benchmark/pytorch_cpu_eager/qwk,█▄▁
benchmark/pytorch_cpu_eager/throughput_ips,▁▃█
+14,...


In [ ]:
summary_rows = []
for ratio, result in e5_pruning_results.items():
    best_backend = min(result["benchmark"]["backends"].items(),
                       key=lambda kv: kv[1]["median_latency_ms_per_image"])
    summary_rows.append({
        "prune_ratio": ratio, "params_after": result["params_after"],
        "pre_finetune_val_macro_f1": result["pre_finetune_val_macro_f1"],
        "post_finetune_val_macro_f1": result["post_finetune_val_macro_f1"],
        "post_finetune_val_qwk": result["post_finetune_val_qwk"],
        "fastest_backend": best_backend[0],
        "fastest_median_latency_ms": best_backend[1]["median_latency_ms_per_image"],
    })
pd.DataFrame(summary_rows)